# B2-023 — Session 5: Latent and Conditional Diffusion

*90 minutes.*

**Program layer:** Round 2 extension  
**Compute:** `compute.policy: cpu` · seed `20261015`  
**Qualified prerequisites:** `book1:F1-scientific-python`, `book1:F3-matrices`, `book1:F4-multivar-calculus`, `book1:F5-probability`, `book1:C1-ml-fundamentals`, `book1:C2-linear-models`, `book1:C5-neural-networks`, `book1:C6-pytorch`, `book1:C11-neural-training`, `B2-019-attention-transformers`, `B2-020-language-transformers`, `B2-021-cross-modal-transformers-vision`, `B2-022-probabilistic-latent-models`  
**Remediation links actually used:** [book1:F1-scientific-python](../../../../book1/units/F1-scientific-python/lesson.ipynb), [book1:F3-matrices](../../../../book1/units/F3-matrices/lesson.ipynb), [book1:F4-multivar-calculus](../../../../book1/units/F4-multivar-calculus/lesson.ipynb), [book1:F5-probability](../../../../book1/units/F5-probability/lesson.ipynb), [book1:C1-ml-fundamentals](../../../../book1/units/C1-ml-fundamentals/lesson.ipynb), [book1:C2-linear-models](../../../../book1/units/C2-linear-models/lesson.ipynb), [book1:C5-neural-networks](../../../../book1/units/C5-neural-networks/lesson.ipynb), [book1:C6-pytorch](../../../../book1/units/C6-pytorch/lesson.ipynb), [book1:C11-neural-training](../../../../book1/units/C11-neural-training/lesson.ipynb), [B2-019-attention-transformers](../../B2-019-attention-transformers/lesson.ipynb), [B2-020-language-transformers](../../B2-020-language-transformers/lesson.ipynb), [B2-021-cross-modal-transformers-vision](../../B2-021-cross-modal-transformers-vision/lesson.ipynb), [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb).

**Data boundary:** CPU only; use literal unit data, the seeded unit generator, and standard library/NumPy/Torch only. Do not use the web, downloads, external datasets, torchvision datasets, model hubs, pretrained models, checkpoints, or student data.

Stable Diffusion is not a new kind of model: it is a composition of parts this course has already built.
It runs Sessions 3–4's diffusion in the latent space of B2-022's autoencoder, lets a text encoder built from B2-019/B2-020's attention and token embeddings steer the denoiser through cross-attention inside B2-021's U-Net, and sharpens the result with classifier-free guidance.
This session teaches each composition step, maps the full pipeline, and trains a faithful CPU-scale analogue: a class-conditional latent diffusion model on 4-D data with a frozen literal encoder and decoder.
Practices after this session: p05, p12, p20, p21, p23, p24.

## 1. Why diffuse in a latent space

Every training step and every one of the $T$ sampling steps runs the denoiser on a tensor of the data's size. For a $512\times512$ RGB image that is $786{,}432$ numbers per step. **Latent diffusion** first compresses the data with an autoencoder and runs diffusion on the much smaller code: Stable Diffusion's codes are $64\times64\times4=16{,}384$ numbers, 48 times fewer.

**The recipe.**

1. Train an `autoencoder` (in Stable Diffusion, a `variational-autoencoder` from [B2-022-probabilistic-latent-models](../../B2-022-probabilistic-latent-models/lesson.ipynb)) so that $\text{decode}(\text{encode}(x))\approx x$.
2. **Freeze** it. Diffusion training sees only $z_0=\text{encode}(x)$ (scaled, Section 2); no gradient ever reaches the autoencoder.
3. Train the diffusion denoiser on latents exactly as in Session 4.
4. To generate: sample a latent with the reverse process, then decode it.

**This unit's frozen autoencoder** is literal, not trained, so that every result is exact and fast. The data set `latent4d` has 240 rows in $\mathbb R^4$ with three class labels (64 training and 16 held-out rows per class). The loader supplies a $(2,4)$ matrix $E$ with **orthonormal rows**, $EE^\top=I_2$ (Book 1 `matrix-multiplication`), and
$$\text{encode}(x)=xE^\top\ \ (B,4)\to(B,2),\qquad \text{decode}(z)=zE\ \ (B,2)\to(B,4).$$
Every stored row was generated as $\text{decode}(z)$ for a 2-D latent $z$, so it lies in the decoder's column space, and then $\text{decode}(\text{encode}(x))=zEE^\top E=zE=x$ **exactly**: the loader checks this in float32 when it is imported. The three class centres in latent space are $(0,3)$, $(-2.5,-1.5)$, $(2.5,-1.5)$, with latent standard deviation $1.25$, so neighbouring centres are $4.0$ to about $4.1$ standard deviations apart (distances $5$ and $\sqrt{26.5}\approx5.15$) and the classes overlap a little.

**Checkpoint 1A.** What are the shapes of `encode(x)` and `decode(encode(x))` for the 192 training rows?

**Checkpoint 1B.** For a general $x\in\mathbb R^4$, $\text{decode}(\text{encode}(x))=xE^\top E$. Why is that not always $x$?

**Collected answers.** `(192,2)` and `(192,4)`. Because $E^\top E$ is a $4\times4$ projection of rank $2$: it keeps the part of $x$ in the decoder's 2-D column space and discards the rest, so only rows already in that space are reproduced exactly.

## 2. The latent scaling factor

Session 3 assumed the diffusion data are of roughly unit scale or smaller: only then is $\sqrt{\bar\alpha_T}\,z_0$ negligible at $t=T$ and starting the sampler from $N(0,I)$ legitimate. Autoencoder latents have whatever scale the autoencoder happened to learn.

**The fix is one constant.** Multiply latents by a fixed **latent scaling factor** $s$ before diffusion, and divide by it before decoding:
$$z_0=s\cdot\text{encode}(x)\ \ \text{(training)},\qquad \hat x=\text{decode}(\hat z_0/s)\ \ \text{(generation)}.$$
Stable Diffusion v1 uses $s=0.18215$, chosen so its latents have roughly unit standard deviation. This unit's literal factor is `LATENT_SCALE = 0.25`: the raw training latents have per-coordinate standard deviations of about $2.2$ and $2.5$, and the scaled ones about $0.56$ and $0.63$.

**Both ends must agree.** Forgetting $s$ at training time trains the denoiser on latents four times too large for the schedule; forgetting $1/s$ at decoding produces samples squeezed four times toward the origin. Either mismatch is silent: the code runs and the loss looks normal. p24 audits a pipeline with exactly this defect.

**Checkpoint 2A.** A sampled latent is $\hat z_0=(0,0.75)$. Which data-space row does the pipeline output with $s=0.25$?

**Checkpoint 2B.** If the decoder forgot to divide by $s$, how far from the origin would the latent of that output be compared with the correct one?

**Collected answers.** $\text{decode}((0,3))$, the decoded centre of class 0, because $\hat z_0/s=(0,3)$. Four times closer: the decoder would receive $(0,0.75)$ instead of $(0,3)$.

In [ ]:
import importlib.util
from pathlib import Path

import torch

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)
import math
from torch import nn

E = generative_data.encoder_matrix()                 # (2, 4), orthonormal rows; frozen, never trained
LATENT_SCALE = generative_data.LATENT_SCALE          # 0.25


def encode(x):
    return x @ E.T                                    # (B, 4) -> (B, 2)


def decode(z):
    return z @ E                                      # (B, 2) -> (B, 4)

x = generative_data.train_tensor("latent4d")
print("E E^T =", (E @ E.T).tolist())
print("exact round trip on every training row:", torch.equal(decode(encode(x)), x))
print("raw latent std   :", [round(v, 3) for v in encode(x).std(dim=0).tolist()])
print("scaled latent std:", [round(v, 3) for v in (encode(x) * LATENT_SCALE).std(dim=0).tolist()])

## 3. Conditioning signals become tokens

A text-to-image model must turn a prompt into something a network can attend to.

* **Text (Stable Diffusion).** The prompt is tokenized, each token becomes a vector by a `learned-token-embedding` (B2-020), and a frozen Transformer text encoder (B2-019's attention stacked into blocks) turns the sequence into **context vectors**, one per token.
* **Class labels (this unit).** Each of the three classes has a learned embedding vector, `class_tokens = nn.Embedding(3, 16)`: a lookup table whose rows are trained like any other weights. A batch of labels of shape `(B,)` becomes token vectors of shape `(B,16)`.
* **The null token.** One extra learned vector, `null_token` of shape `(16,)`, means "no condition". Section 5 explains why the network must also learn to work with the null token alone; in Stable Diffusion its role is played by the encoding of the empty prompt.

Every example therefore offers the denoiser a short sequence of **condition tokens**: here, its class token followed by the null token, shape `(B,2,16)`.

**Checkpoint 3A.** What is the shape of `class_tokens(labels)` for 192 labels?

**Checkpoint 3B.** Why is the null token a learned parameter rather than a fixed zero vector?

**Collected answers.** `(192,16)`. So the model can learn what "no condition" should contribute; after the key and value projections a fixed zero vector would give a zero key and value, a fixed choice the model could not adapt.

## 4. Cross-attention conditioning

**Cross-attention** is B2-019's `query-key-value-attention` with the two roles split between modalities, following B2-021's **modal-ownership rule**: the modality being updated owns the queries, and the modality supplying information owns the keys and values.

* **Queries from the latent.** Here the latent being denoised is one token: a linear projection of $(z_t,\,t/T)$, so $q=W_q[z_t;t/T]+b_q$ with `q_proj = nn.Linear(3, 16)`, shape `(B,1,16)`.
* **Keys and values from the condition tokens.** `k_proj` and `v_proj` are `nn.Linear(16, 16, bias=False)` applied to the `(B,2,16)` condition tokens.
* **Attention.** Scores $qk^\top/\sqrt{16}$ of shape `(B,1,2)`, a softmax over the two condition tokens (Book 1 `softmax`), and the output $c$ is the weighted sum of the values, shape `(B,16)` after removing the singleton query axis.

**Shape ledger (prose):** query `(B,1,16)`; tokens `(B,2,16)`; keys `(B,2,16)`; values `(B,2,16)`; scores `(B,1,2)`; weights `(B,1,2)`; output `(B,16)`. The output count equals the query count, as B2-021's rule predicts.

**Removing the class token.** For an example that should be unconditioned, the class token's score is set to $-\infty$ before the softmax. Its weight is then exactly $0$, the null token's weight exactly $1$, and $c=v_\varnothing$: identical to attending over the null token alone. Setting the score to $0$ instead would **not** remove the token.

**In Stable Diffusion** the same block sits inside the `unet` denoiser of B2-021 at several resolutions: every spatial position of the latent feature map is a query, and the context vectors of the prompt supply keys and values.

**Worked example.** Suppose after projection $q=(1,0)$, the class key and value are $k_c=(2,0)$ and $v_c=(1,1)$, and the null key and value are $k_\varnothing=(0,1)$ and $v_\varnothing=(0,-1)$, with $d=2$. Scores: $(2/\sqrt2,\,0)=(1.414214,0)$. Weights: $\frac{e^{1.414214}}{e^{1.414214}+1}=0.804429$ and $0.195571$. Output: $0.804429(1,1)+0.195571(0,-1)=(0.804429,0.608858)$. With the class token removed, the output is $v_\varnothing=(0,-1)$.

**Checkpoint 4A.** In this unit's block, which modality owns the queries and which owns the keys and values?

**Checkpoint 4B.** In the worked example, what would the output be if the class token's score were set to $0$ instead of $-\infty$?

**Collected answers.** The latent (with its timestep feature) owns the queries; the condition tokens (class token and null token) own the keys and values. Both scores would be $0$, the weights $(\tfrac12,\tfrac12)$, and the output $\tfrac12(1,1)+\tfrac12(0,-1)=(0.5,0)$: the class information would still leak in.

## 5. Conditioning dropout and classifier-free guidance

**Conditioning dropout (training).** With probability $p_{\text{drop}}=0.2$, independently for each training example and each update, the class token is removed (Section 4), so the denoiser sees the null token only. One network thus learns two predictions: the **conditional** $\varepsilon_\theta(z_t,t,c)$ and the **unconditional** $\varepsilon_\theta(z_t,t,\varnothing)$. The drop decisions are drawn from the training loop's seeded generator, like `t` and `eps`.

**Classifier-free guidance (sampling).** At every reverse step, evaluate both predictions and combine them:
$$\hat\varepsilon=\varepsilon_{\text{uncond}}+w\,\big(\varepsilon_{\text{cond}}-\varepsilon_{\text{uncond}}\big).$$

* $w=0$ gives the unconditional model: samples from all classes.
* $w=1$ gives the conditional model.
* $w>1$ **extrapolates** past the conditional prediction, away from the unconditional one. Up to sign and scale, a noise prediction points against the gradient of the log-density, and the difference $\varepsilon_{\text{cond}}-\varepsilon_{\text{uncond}}$ corresponds to the gradient of $\log p(c\mid z_t)$ (stated, not derived). Guidance therefore pushes samples toward regions where the class is more certain: fewer ambiguous samples between classes, less diversity.
* Too large a $w$ **overshoots**: samples are pushed past the class's typical region, away from the other classes. The cost of any guidance is two network evaluations per step.

The unconditional prediction uses exactly the training-time null-only path. If dropout were never applied in training, that path would be untrained and guidance would amplify noise. Implementing `cfg_combine(eps_uncond, eps_cond, w)` is p12.

**Worked exam-style example.** With $\varepsilon_{\text{uncond}}=(0.2,-0.1)$ and $\varepsilon_{\text{cond}}=(0.6,0.3)$, the difference is $(0.4,0.4)$, and $w=0,1,1.5,2$ give $(0.2,-0.1)$, $(0.6,0.3)$, $(0.8,0.5)$, $(1.0,0.7)$. A sign-flipped implementation $\varepsilon_{\text{uncond}}-w(\varepsilon_{\text{cond}}-\varepsilon_{\text{uncond}})$ agrees only at $w=0$; at $w=1$ it returns $(-0.2,-0.5)$, guiding **away** from the class.

**Checkpoint 5A.** Compute $\hat\varepsilon$ for $\varepsilon_{\text{uncond}}=(1,0)$, $\varepsilon_{\text{cond}}=(0,1)$, and $w=3$.

**Checkpoint 5B.** Why must conditioning dropout be applied during training for guidance to work?

**Collected answers.** $(1,0)+3(-1,1)=(-2,3)$. Because $\varepsilon_{\text{uncond}}$ comes from the null-only path; only training examples whose class token was dropped teach the network that path.

## 6. The Stable Diffusion pipeline component map

| Component | Role | Course source | This unit's CPU analogue |
|---|---|---|---|
| Text encoder | turns the prompt into context tokens (keys/values for cross-attention); frozen | B2-020 `learned-token-embedding`, B2-019 attention | `nn.Embedding(3, 16)` class tokens plus a learned null token |
| Denoiser | predicts $\varepsilon$ from $(z_t,t,\text{context})$; the only network trained during diffusion training | B2-021 `unet` with cross-attention blocks (B2-019 `query-key-value-attention`) | an MLP on $(z_t,t/T,c)$, with $c$ from one single-head cross-attention block |
| VAE encoder / decoder | maps images to latents and back; frozen | B2-022 `autoencoder`, `variational-autoencoder` | literal orthonormal `encode`/`decode` with `LATENT_SCALE = 0.25` |
| Scheduler | the $\beta$, $\alpha$, $\bar\alpha$, $\tilde\beta$ tables, the training noise (`q_sample`), and the reverse update (`p_sample`, `sample_loop`) | Sessions 3–4 (`multivariate-gaussian` noise, `gaussian-reparameterization`) | `make_schedule(50, 1e-3, 0.2)` and the ancestral sampler |
| Guidance | combines conditional and unconditional predictions | Section 5 | `cfg_combine` with the null-only path |

**Training data flow.** image → VAE encoder → $\times s$ → `q_sample` with a random $t$ and seeded $\varepsilon$ → denoiser (queries from the noisy latent, keys/values from the prompt's context, dropped to null with probability $p_{\text{drop}}$) → $\lVert\varepsilon-\hat\varepsilon\rVert^2$.

**Inference data flow.** prompt → text encoder → context; $z_T\sim N(0,I)$ → for $t=T,\dots,1$: two denoiser calls (context and null), guidance, one scheduler step → $\hat z_0$ → $\div s$ → VAE decoder → image.

The `kl-divergence` regularizer of the VAE matters only when the autoencoder itself is trained; once frozen, diffusion training never sees it.

**Checkpoint 6A.** Which component is trained during diffusion training, and which are frozen?

**Checkpoint 6B.** In the inference flow, how many denoiser evaluations does one sampling step with guidance need?

**Collected answers.** Only the denoiser (with its cross-attention and, here, the class and null tokens) is trained; the text encoder and the VAE are frozen. Two: one with the prompt's context and one with the null context.

## 7. Worked laboratory: a CPU-scale class-conditional latent diffusion trace

The cell below builds the unit's analogue end to end at lesson scale:

* the frozen `encode`/`decode` and `LATENT_SCALE` from Section 2, applied once to the 192 training rows;
* a cross-attention conditioner (Section 4: `q_proj`, `k_proj`, `v_proj`, `class_tokens`, `null_token`, created in that order) feeding an MLP `19→64→64→2` on $(z_t,t/T,c)$;
* Session 4's schedule and `q_sample`, conditioning dropout $0.2$, Adam with `lr=1e-2`, and **800** full-batch updates whose `t`, `eps`, and drop decisions come from one dedicated generator;
* guided sampling of 300 latents (100 per class) from one fixed set of starting noise and step noises, reused for every $w$.

For each $w$ it reports the **mean distance** from the decoded samples to their target class centre in data space and the **hit rate**, the fraction whose nearest class centre (lowest index on ties, as in Session 2) is the target.

**What to look for.** At $w=0$ the samples ignore the label: the hit rate is about one third and the mean distance is large. At $w=1$ nearly every sample lands on its class. At $w=1.5$ the remaining ambiguous samples are cleaned up: the mean distance drops further and the hit rate reaches $1$. At $w=3$ guidance overshoots: the hit rate stays at $1$, but the mean distance **rises** above its $w=1.5$ value (in this short lesson-scale run, even above $w=1$), because samples are pushed past their class centres away from the other classes. Hit rates alone would miss this; the continuous distance shows it. p20 runs the pinned 1,500-update protocol and certifies this pattern for $w=0,1,1.5$.

**Checkpoint 7A.** Why are the same $x_T$ and step noises reused for every $w$?

**Checkpoint 7B.** Why does p20 certify a continuous mean distance in addition to hit rates?

**Collected answers.** So that the only difference between the runs is the guidance weight; otherwise differences in the random draws would be mixed into the comparison. Hit rates saturate at $1$ (two values of $w$ can tie), and they cannot see overshoot or within-class drift; the mean distance changes continuously and can show strict improvement.

In [ ]:
import importlib.util
from pathlib import Path

import torch

SEED = 20261015
LOADER_CANDIDATES = (
    Path("../data/generative_data.py"),
    Path("units/B2-023-generative-models-diffusion/data/generative_data.py"),
)
loader_path = next(path for path in LOADER_CANDIDATES if path.is_file())
loader_spec = importlib.util.spec_from_file_location("generative_data", loader_path)
generative_data = importlib.util.module_from_spec(loader_spec)
loader_spec.loader.exec_module(generative_data)
torch.set_num_threads(1)
import math
from torch import nn

E = generative_data.encoder_matrix()                 # (2, 4), orthonormal rows; frozen, never trained
LATENT_SCALE = generative_data.LATENT_SCALE          # 0.25


def encode(x):
    return x @ E.T                                    # (B, 4) -> (B, 2)


def decode(z):
    return z @ E                                      # (B, 2) -> (B, 4)
import torch.nn.functional as F

T = 50
beta = torch.zeros(T + 1, dtype=torch.float64)
beta[1:] = torch.linspace(1e-3, 0.2, T, dtype=torch.float64)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)
beta_tilde = torch.zeros(T + 1, dtype=torch.float64)
beta_tilde[2:] = (1 - alpha_bar[1:-1]) / (1 - alpha_bar[2:]) * beta[2:]


class Conditioner(nn.Module):
    def __init__(self, query_in=3, dim=16, num_classes=3):
        super().__init__()
        self.dim = dim
        self.q_proj = nn.Linear(query_in, dim)
        self.k_proj = nn.Linear(dim, dim, bias=False)
        self.v_proj = nn.Linear(dim, dim, bias=False)
        self.class_tokens = nn.Embedding(num_classes, dim)
        self.null_token = nn.Parameter(torch.zeros(dim))

    def forward(self, query_feats, labels, keep):
        B = query_feats.shape[0]
        q = self.q_proj(query_feats).unsqueeze(1)                                        # (B,1,dim)
        tokens = torch.stack([self.class_tokens(labels), self.null_token.expand(B, self.dim)], dim=1)  # (B,2,dim)
        scores = q @ self.k_proj(tokens).transpose(1, 2) / math.sqrt(self.dim)            # (B,1,2)
        drop_class = torch.stack([~keep, torch.zeros_like(keep)], dim=1).unsqueeze(1)    # (B,1,2)
        weights = torch.softmax(scores.masked_fill(drop_class, float("-inf")), dim=-1)
        return (weights @ self.v_proj(tokens)).squeeze(1)                                 # (B,dim)


class Denoiser(nn.Module):
    def __init__(self):
        super().__init__()
        self.cond = Conditioner()
        self.mlp = nn.Sequential(nn.Linear(19, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 2))

    def forward(self, z_t, t_scaled, labels, keep):
        c = self.cond(torch.cat([z_t, t_scaled], dim=1), labels, keep)
        return self.mlp(torch.cat([z_t, t_scaled, c], dim=1))


rows = generative_data.train_tensor("latent4d")
labels = generative_data.train_label_tensor("latent4d")
z0 = encode(rows) * LATENT_SCALE
torch.manual_seed(SEED)
model = Denoiser()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-2)
g = torch.Generator().manual_seed(SEED + 20)
trace = []
for _ in range(800):
    t = torch.randint(1, T + 1, (z0.shape[0],), generator=g)
    eps = torch.randn(z0.shape[0], 2, generator=g)
    keep = torch.rand(z0.shape[0], generator=g) >= 0.2
    ab = alpha_bar[t].to(z0.dtype).unsqueeze(1)
    z_t = ab.sqrt() * z0 + (1 - ab).sqrt() * eps
    optimizer.zero_grad(set_to_none=True)
    loss = F.mse_loss(model(z_t, (t.to(torch.float32) / T).unsqueeze(1), labels, keep), eps)
    loss.backward()
    optimizer.step()
    trace.append(loss.item())
print("loss window means every 200 steps:", [round(sum(trace[a:a + 50]) / 50, 3) for a in range(0, 800, 200)])

targets = torch.arange(3).repeat_interleave(100)
sample_generator = torch.Generator().manual_seed(SEED + 21)
z_T = torch.randn(300, 2, generator=sample_generator)
noises = torch.randn(T, 300, 2, generator=sample_generator)
on, off = torch.ones(300, dtype=torch.bool), torch.zeros(300, dtype=torch.bool)
centers = generative_data.class_centers()
for w in (0.0, 1.0, 1.5, 3.0):
    with torch.no_grad():
        z = z_T
        for t in range(T, 0, -1):
            ts = torch.full((300, 1), t / T)
            e_u, e_c = model(z, ts, targets, off), model(z, ts, targets, on)
            eps_hat = e_u + w * (e_c - e_u)
            mean = (z - beta[t].item() / math.sqrt(1 - alpha_bar[t].item()) * eps_hat) / math.sqrt(alpha[t].item())
            z = mean + math.sqrt(beta_tilde[t].item()) * noises[t - 1] if t >= 2 else mean
        x = decode(z / LATENT_SCALE)
    distance = (x - centers[targets]).norm(dim=1).mean().item()
    hit = (((x[:, None, :] - centers[None]) ** 2).sum(dim=2).argmin(dim=1) == targets).float().mean().item()
    print(f"w={w:3.1f}: mean distance to target centre {distance:.3f}   hit rate {hit:.3f}")

## 8. Common pitfalls and Exam connections

**Common pitfalls (each with its fix).**

1. *Sign-flipped guidance.* `eps_uncond - w * (eps_cond - eps_uncond)` guides away from the class and agrees with the correct formula only at $w=0$. Fix: probe at $w=1$, where the result must equal `eps_cond`.
2. *Missing latent scaling at one end.* Fix: multiply by $s$ before `q_sample`, divide by $s$ before `decode`, and test the round trip.
3. *No conditioning dropout.* The null path is never trained, so $\varepsilon_{\text{uncond}}$ is garbage. Fix: drop with probability $p_{\text{drop}}$ from the seeded generator.
4. *"Removing" a token by zeroing its score.* A zero score still receives softmax weight. Fix: mask with $-\infty$.
5. *Reversed ownership.* Queries from the condition tokens and keys/values from the latent produce one output per condition token, not per latent token. Fix: latent owns $Q$; condition owns $K,V$.
6. *Leakage through the conditioning set.* Building class embeddings, prototypes, or a validation choice of $w$ from held-out rows. Fix: training rows only enter the frozen `encode` during training, and $w$ is chosen without held-out rows.
7. *Too much guidance.* Large $w$ overshoots; read a continuous metric, not only hit rates.

**Exam connections.** p05 matches pipeline components to their roles; p12 implements `cfg_combine` and the cross-attention block with an exact shape ledger and literal probes; p20 trains the CPU-scale analogue and certifies the guidance pattern; p21 chooses among GANs, VAEs, and diffusion under stated constraints; p24 audits a pipeline with a sign-flipped guidance term, a missing scaling factor, and leakage in its conditioning set.

**Checkpoint 8A.** A guidance implementation returns `eps_uncond` at $w=1$. Is it correct?

**Checkpoint 8B.** Which practice audits a pipeline for leakage in its conditioning set?

**Collected answers.** No: at $w=1$ correct guidance returns `eps_cond`. p24.

## 9. Forward-only deeper topic: real Stable Diffusion on a GPU

Forward-only deeper topic.
Running the real Stable Diffusion means downloading pretrained weights for its text encoder, U-Net, and VAE from a model hub and sampling $64\times64\times4$ latents on a GPU, usually with a faster scheduler than ancestral sampling.
That workflow needs hardware, network access, and pretrained weights, all outside this unit's CPU-only, from-scratch boundary; it belongs to the GPU capstone `B2-024`.
None of it is a prerequisite, a permitted shortcut, or an assessed API in B2-023.

This final section is forward-only and not assessed.

**Checkpoint 9A.** Does any B2-023 practice load pretrained Stable Diffusion weights?

**Checkpoint 9B.** Which unit owns the GPU workflow?

**Collected answers.** No; every practice trains a CPU-scale analogue from scratch, and this section is forward-only and not assessed. The GPU capstone, B2-024.